In [37]:
import tables
import numpy as np
from ctapipe_io_lst.constants import LST1_LOCATION as location
from datetime import datetime, timezone

import requests
from astropy.time import Time

import json

#a = tables.open_file("/data/cta/users-ifae/moralejo/CTA/summer_student_2026/datacheck/datacheck_dl1_LST-1.Run24704.h5")
a = tables.open_file("/data/cta/users-ifae/moralejo/CTA/summer_student_2026/datacheck/ALL_h5/datacheck_dl1_LST-1.Run20757.h5")

In [28]:
# we'll use /fov/satellite-passes/ from SatChecker webpage https://satchecker.cps.iau.org/api/docs/#/Field%20of%20View/get_fov_satellite_passes_
"""
# requested parameters (there are more): 
- lat (deg) , long (deg), elevation (m)
- duration of the observation (s)
- ra, dec : Right ascension / Declination of field center in decimal degrees
- fov_radius : Radius of field of view in decimal degrees
- start_time_jd : Start time of observation in Julian date (either this or mid_obs_time_jd must be provided)
- mid_obs_time_jd: Mid-observation time in Julian date (either this or start_time_jd must be provided)
- group_by: Group results by 'satellite' or 'time' (default is 'time' for chronological order)
- illuminated_only: Whether to include only illuminated satellites (default is false)
- async: Whether to process the request asynchronously. If true or omitted, returns a task ID for polling status. If false, returns immediate results.
"""

"\n# requested parameters (there are more): \n- lat (deg) , long (deg), elevation (m)\n- duration of the observation (s)\n- ra, dec : Right ascension / Declination of field center in decimal degrees\n- fov_radius : Radius of field of view in decimal degrees\n- start_time_jd : Start time of observation in Julian date (either this or mid_obs_time_jd must be provided)\n- mid_obs_time_jd: Mid-observation time in Julian date (either this or start_time_jd must be provided)\n- group_by: Group results by 'satellite' or 'time' (default is 'time' for chronological order)\n- illuminated_only: Whether to include only illuminated satellites (default is false)\n- async: Whether to process the request asynchronously. If true or omitted, returns a task ID for polling status. If false, returns immediate results.\n"

In [39]:
print( np.degrees(np.arctan(1/28)))
print(a.root.dl1datacheck.cosmics.col('tel_ra')[8] )
print(a.root.dl1datacheck.cosmics.col('tel_dec')[8] )

2.0454084888872277
288.59345667083386
10.386518391485453


In [40]:
# SUBRUN NUMBERS OF INTEREST
subrun = 98

# TEL RA AND DEC (degrees)
tel_ra = a.root.dl1datacheck.cosmics.col('tel_ra')[subrun] 
tel_dec = a.root.dl1datacheck.cosmics.col('tel_dec')[subrun]

# DRAGON TIMES (UTC for 50 events through the run, given in Unix time)
# Unix time = seconds transcurred from January 1st 1970 at 00:00:00 UTC
ti_unix = min(a.root.dl1datacheck.cosmics.col('dragon_time')[subrun])
tmean_unix = np.mean(a.root.dl1datacheck.cosmics.col('dragon_time')[subrun])
tf_unix = max(a.root.dl1datacheck.cosmics.col('dragon_time')[subrun])

duration = tf_unix - ti_unix

# Conversion to Julian date
ti_jd = Time(ti_unix, format='unix', scale='utc').jd
tmean_jd = Time(tmean_unix, format='unix', scale='utc').jd

# FoV radius: alpha/2 = arctan (d/2f)  
fov_radius = 4.3/2 # source : https://www.ctao.org/emission-to-discovery/telescopes/lst/

# Ask SatChecker API
url = "https://satchecker.cps.iau.org/fov/satellite-passes/"
params = {
    "latitude": location.lat.deg,
    "longitude": location.lon.deg,
    "elevation": location.height.to_value('m'),
    "duration": duration,
    "ra": tel_ra,
    "dec": tel_dec,
    "fov_radius": fov_radius,  # LST1 Field of View radius in degrees
    "start_time_jd": ti_jd,
    "group_by": 'satellite',
    "illuminated_only": 'false',
    "async": 'false'
}

response = requests.get(url, params=params)  

if response.status_code == 200:
    payload = response.json()
    data = payload.get("data", {})

    satellites = data.get("satellites", data) if isinstance(data, dict) else {}

    if not satellites:
        print("No satellites found in this subrun.")
    else:
        print(f"Satellites detected in FOV ({len(satellites)}):")
        for key, sat in satellites.items():
            positions = sat.get("positions", [])
            print(f"- NORAD ID: {sat.get('norad_id')} | Name: {sat.get('name')}")
            print("Duration of satellite obs with LST-1: ", datetime.fromtimestamp(ti_unix, tz=timezone.utc), " - ", datetime.fromtimestamp(tf_unix, tz=timezone.utc))
            print("SatCheck time of satellite in FoV: ",  positions[0].get('date_time') if positions else None, " - ", positions[-1].get('date_time'))

            radius_values = []
            for p in positions:
                radius_values.append(p.get('range_km'))
            print("LST1 - Satellite mean distance: ",  np.mean(radius_values), ' km')
else:
    print(f"Error querying database: HTTP {response.status_code}")
    print("Details:", response.text)



Satellites detected in FOV (2):
- NORAD ID: 62007 | Name: FALCON 9 R/B
Duration of satellite obs with LST-1:  2025-06-22 03:40:29.517793+00:00  -  2025-06-22 03:40:37.124123+00:00
SatCheck time of satellite in FoV:  2025-06-22 03:40:29 UTC  -  2025-06-22 03:40:36 UTC
LST1 - Satellite mean distance:  37195.20194183125  km
- NORAD ID: 19557 | Name: SL-6 R/B(2)
Duration of satellite obs with LST-1:  2025-06-22 03:40:29.517793+00:00  -  2025-06-22 03:40:37.124123+00:00
SatCheck time of satellite in FoV:  2025-06-22 03:40:29 UTC  -  2025-06-22 03:40:36 UTC
LST1 - Satellite mean distance:  17764.01831291625  km


In [ ]:
"""
{
  "data": {
    "satellites": {
      "additionalProp1": {
        "name": "string",
        "norad_id": 0,
        "positions": [
          {
            "altitude": 0,
            "angle": 0,
            "azimuth": 0,
            "date_time": "string",
            "dec": 0,
            "julian_date": 0,
            "orbital_data": {
              "source": "string",
              "tle_line1": "string",
              "tle_line2": "string"
            },
            "orbital_data_epoch": "string",
            "orbital_data_source": "tle",
            "ra": 0,
            "range_km": 0
          }
        ]
"""